In [1]:
import pandas as pd

In [2]:
# CONFIGURATION

METADATA_FILE = "../data/country_metadata.csv"
LIFE_TOTAL_FILE = "../data/total_life_expectancy_at_birth.csv"
LIFE_MALE_FILE = "../data/male_life_expectancy_at_birth.csv"
LIFE_FEMALE_FILE = "../data/female_life_expectancy_at_birth.csv"
DEATH_RATE_FILE = "../data/death_rate_crude.csv"
FERTILITY_RATE_FILE = "../data/fertility_rate_total.csv"

In [3]:
# Helper function


def reshape_indicator(df, value_column_name):
    year_columns = []
    for col in df.columns:
        if str(col).isdigit():
            year_columns.append(col)

    long_df = pd.melt(
        df,
        id_vars=["Country Name", "Country Code"],
        value_vars=year_columns,
        var_name="Year",
        value_name=value_column_name,
    )
    long_df["Year"] = long_df["Year"].astype(int)
    return long_df

In [4]:
# Read metadata

metadata = pd.read_csv(METADATA_FILE)

# Dropping the entries with blank IncomeGroup countries - these are the unions, aggregations etc
metadata = metadata[metadata["IncomeGroup"].notna()]


# Read indicator files

life_total_raw = pd.read_csv(LIFE_TOTAL_FILE, skiprows=4)
life_male_raw = pd.read_csv(LIFE_MALE_FILE, skiprows=4)
life_female_raw = pd.read_csv(LIFE_FEMALE_FILE, skiprows=4)
death_rate_raw = pd.read_csv(DEATH_RATE_FILE, skiprows=4)
fertility_rate_raw = pd.read_csv(FERTILITY_RATE_FILE, skiprows=4)

# Return only valid countries

valid_country_codes = metadata["Country Code"].unique()

life_total_raw = life_total_raw[
    life_total_raw["Country Code"].isin(valid_country_codes)
]
life_male_raw = life_male_raw[
    life_male_raw["Country Code"].isin(valid_country_codes)
]
life_female_raw = life_female_raw[
    life_female_raw["Country Code"].isin(valid_country_codes)
]
death_rate_raw = death_rate_raw[
    death_rate_raw["Country Code"].isin(valid_country_codes)
]
fertility_rate_raw = fertility_rate_raw[
    fertility_rate_raw["Country Code"].isin(valid_country_codes)
]

In [5]:
# Convert to long format

life_total = reshape_indicator(life_total_raw, "LifeExpectancy")
life_male = reshape_indicator(life_male_raw, "MaleLifeExpectancy")
life_female = reshape_indicator(life_female_raw, "FemaleLifeExpectancy")
death_rate = reshape_indicator(death_rate_raw, "DeathRate")
fertility_rate = reshape_indicator(fertility_rate_raw, "FertilityRate")

In [6]:
# Merge all indicators

dashboard_df = (
    life_total.merge(
        life_male, on=["Country Name", "Country Code", "Year"], how="left"
    )
    .merge(
        life_female, on=["Country Name", "Country Code", "Year"], how="left"
    )
    .merge(death_rate, on=["Country Name", "Country Code", "Year"], how="left")
    .merge(
        fertility_rate, on=["Country Name", "Country Code", "Year"], how="left"
    )
)

In [7]:
dashboard_df

,Country Name,Country Code,Year,LifeExpectancy,MaleLifeExpectancy,FemaleLifeExpectancy,DeathRate,FertilityRate
0,Aruba,ABW,1960,64.049,60.746,67.459,7.525,4.567
1,Afghanistan,AFG,1960,32.799,32.136,33.549,31.672,7.282
2,Angola,AGO,1960,37.933,36.248,39.739,27.055,6.708
3,Albania,ALB,1960,56.413,54.005,58.877,15.888,6.383
4,Andorra,AND,1960,72.094,69.109,75.081,7.066,2.545
...,...,...,...,...,...,...,...,...
14317,Kosovo,XKX,2025,NaN,NaN,NaN,NaN,NaN
14318,"Yemen, Rep.",YEM,2025,NaN,NaN,NaN,NaN,NaN
14319,South Africa,ZAF,2025,NaN,NaN,NaN,NaN,NaN
14320,Zambia,ZMB,2025,NaN,NaN,NaN,NaN,NaN


In [8]:
# Create Gender gap
# Positive => Female higher
# Negative => Make higher


dashboard_df["GenderGap"] = (
    dashboard_df["FemaleLifeExpectancy"] - dashboard_df["MaleLifeExpectancy"]
)

In [9]:
# Add income group and region

metadata_subset = metadata[["Country Code", "IncomeGroup", "Region"]]
dashboard_df = dashboard_df.merge(
    metadata_subset, on="Country Code", how="left"
)

In [10]:
# Sort table
dashboard_df.sort_values(by=["Country Name", "Year"])

# Remove rows of Year 2025 as they are all empty

dashboard_df = dashboard_df[dashboard_df["Year"] != 2025]

In [11]:
# Save outout table

dashboard_df.to_csv("dashboard_df.csv", index=False)

In [12]:
dashboard_df

,Country Name,Country Code,Year,LifeExpectancy,MaleLifeExpectancy,FemaleLifeExpectancy,DeathRate,FertilityRate,GenderGap,IncomeGroup,Region
0,Aruba,ABW,1960,64.049,60.746,67.459,7.525,4.567,6.713,High income,Latin America & Caribbean
1,Afghanistan,AFG,1960,32.799,32.136,33.549,31.672,7.282,1.413,Low income,Middle East & North Africa
2,Angola,AGO,1960,37.933,36.248,39.739,27.055,6.708,3.491,Lower middle income,Sub-Saharan Africa
3,Albania,ALB,1960,56.413,54.005,58.877,15.888,6.383,4.872,Upper middle income,Europe & Central Asia
4,Andorra,AND,1960,72.094,69.109,75.081,7.066,2.545,5.972,High income,Europe & Central Asia
...,...,...,...,...,...,...,...,...,...,...,...
14100,Kosovo,XKX,2024,78.222,76.008,80.281,5.973,1.538,4.273,Upper middle income,Europe & Central Asia
14101,"Yemen, Rep.",YEM,2024,69.439,67.366,71.549,4.734,4.499,4.183,Low income,Middle East & North Africa
14102,South Africa,ZAF,2024,66.312,62.783,69.792,9.237,2.205,7.009,Upper middle income,Sub-Saharan Africa
14103,Zambia,ZMB,2024,66.528,64.100,68.874,5.165,4.036,4.774,Lower middle income,Sub-Saharan Africa
